<h3>LinkIt Grade Cleanup </h3>
This script processes student grades from a CSV file exported from Powerschool SIS, filters and cleans the student grades and prepares a new CSV file (Stored_Grades.csv) for delivery to Linkit!
<br><br>
Note:

* Modify the script as needed to adjust filtering criteria, file paths, and conversion logic.
* Consider uncommenting the line for filling missing Letter Grades with corresponding values from Numeric Grade if needed.

<b>Import libraries:</b>
* pandas: Used for data manipulation (reading and writing CSV files).
* os: Used to check and potentially create the output directory.

In [4]:
import pandas as pd
import os

<b>Set file paths:</b>
* Update the manual file paths to wherever you need or utilize the OS calls to automatically set the working directory to the users cureent active directory
* Read student grade CSV into DataFrame 'data'
* Check for active directory and create it if it doesn't exist

In [6]:
#Update the manual file paths to wherever you need
#dir = r"C:\Users\[Username]\Downloads"
#data_path = r"C:\Users\[Username]\Downloads\linkit_stored_grades.csv"
#output_path = r"C:\Users\[Username]\Documents\[SubDirectories]\Stored_Grades.csv"

#Automatic Directory Retrieval
# Get the user's current working directory
dir = os.getcwd()

# Combine the current path and file name using os.path.join
data_path = os.path.join(dir, "linkit_stored_grades.csv")
output_path = os.path.join(dir, "clean_grades.csv")

data = pd.read_csv(data_path)

# Check if the data and directories exists
if os.path.exists(data_path):
  # Create the directory if it doesn't exist
  data = pd.read_csv(data_path)
  print("File", data_path, "uploaded successfully!")
else:
  print("File", data_path, "not found!")


File C:\Users\rkroker\Documents\Code\Python\Linkit Grades\linkit_stored_grades.csv uploaded successfully!


<b> Grade Conversions </b><br>
This section of code prepares for converting letter grades based on grade level. It defines lists to exclude unwanted courses and specific grades from processing. Then, it creates separate dictionaries for middle school (MS) and high school (PHS) that map numeric grade ranges to letter grades. Finally, a function convert_grade is defined to handle the conversion process. It attempts to convert the grade to a number, then uses the grade level to pick the appropriate conversion dictionary (MS or PHS) and applies the conversion based on the grade range. The function ensures any errors during conversion (like non-numeric grades) are handled by returning the original value.

In [8]:
# List of courses to exclude
excluded_courses = ["MS_HR", "-"]

# List of Grades to exclude
excluded_letter_grades = ["NG", "ME", "INC", "WP", "WF", "-"]
excluded_numeric_grades = ["0", "-"]

# Define a dictionary to map numeric grades to letter grades
MS_conversions = {
    range(0, 60): 'F',
    range(60, 70): 'D',
    range(70, 80): 'C',
    range(80, 90): 'B',
    range(90, 500): 'A'
}
PHS_conversions = {
    range(0, 60): 'F',
    range(60, 63): 'D-',
    range(63, 67): 'D',
    range(67, 70): 'D+',
    range(70, 73): 'C-',
    range(73, 77): 'C',
    range(77, 80): 'C+',
    range(80, 83): 'B-',
    range(83, 87): 'B',
    range(87, 90): 'B+',
    range(90, 93): 'A-',
    range(93, 97): 'A',
    range(97, 100): 'A+'
}

# Function to convert grade based on grade level
def convert_grade(grade, level):
  try:
    # Attempt to convert grade to float (numeric grades will succeed)
    grade = int(grade)
    if level in range(6, 9):  # Middle school grades (inclusive)
        if grade < 60:
            return 'F'
        elif grade < 70:
            return 'D'
        elif grade < 80:
            return 'C'
        elif grade < 90:
            return 'B'
        else:
            return 'A'
    elif level in range(9, 13):  # High school grades (inclusive)
        if grade < 60:
            return 'F'
        elif grade < 63:
            return 'D-'
        elif grade < 67:
            return 'D'
        elif grade < 70:
            return 'D+'
        elif grade < 73:
            return 'C-'
        elif grade < 77:
            return 'C'
        elif grade < 80:
            return 'C+'
        elif grade < 83:
            return 'B-'
        elif grade < 87:
            return 'B'
        elif grade < 90:
            return 'B+'
        elif grade < 93:
            return 'A-'
        elif grade < 97:
            return 'A'
        else:
            return 'A+'
    else:
        # Handle potential invalid grade levels (optional)
        return grade
  except (ValueError, TypeError):
    # If conversion fails (non-numeric grades), return the original value
    return grade


<b>Filtering and Conversions</b><br>
This code block cleans and prepares the data for final letter grade conversion. First, it filters the data to only include rows with the final term ("Y1" in this case) and removes rows based on unwanted courses and specific letter grades. It then tackles missing values in the "Letter Grade" column. While there's a commented-out approach to fill blanks with corresponding numeric grades, the active code focuses on removing any leading/trailing spaces and filling all remaining blanks (including spaces) with converted letter grades using the previously defined convert_grade function. This function is applied efficiently across all rows using vectorized operations.

In [13]:
data.head()

,School Year,Termbinsname,Schoolname,Schoolid,Studentlastname,Studentfirstname,Student Number,State Studentnumber,Grade Level,Course Number,Course Name,Sectionid,Credit Type,Teacher,Teacherlastname,Teacherfirstname,Percent,Grade
0,2023-2024,Y1,Charles Boehm Middle School,255,Pringle,Anthony,1600101,5066239294,8,MATH180,Math 180,29616,MA,3359,Dale,Ashley,90.66,A
1,2023-2024,Y1,Charles Boehm Middle School,255,Roberts,Mikayla,1700142,9613941533,7,MATH180,Math 180,27884,MA,3359,Dale,Ashley,94.59,A
2,2023-2024,Y1,Charles Boehm Middle School,255,Hess,McKenzie,1700217,6513576563,7,MATH180,Math 180,27976,MA,3359,Dale,Ashley,89.53,A
3,2023-2024,Y1,Charles Boehm Middle School,255,Minton,Kaitlyn,1800473,5166477251,6,MATH6,Math 6th,28343,MA,2532,Murphy,David,94.17,A
4,2023-2024,Y1,Charles Boehm Middle School,255,Marota,Mira,1800778,6211658575,8,MATH8,Math 8th,28346,MA,3806,Diamond,Debra,93.89,A


In [17]:
filtered_data = data[
    (data["Termbinsname"] == "Y1")  # Filter by term
    & (~data["Course Number"].isin(excluded_courses))  # Exclude specific courses
    & (~data["Course Number"].str.startswith(("AR", "AE", "EL", "FCS", "MU", "SP", "PE", "JROTC")))  # Exclude specific prefixes
    & (~data["Percent"].isin(excluded_numeric_grades))
    & (~data["Grade"].isin(excluded_letter_grades))  # Exclude specific letter grades
]

# Apply convert_grade function to each row, replacing 'Letter Grade' column with converted values
filtered_data.loc[:, "Grade"] = filtered_data.apply(lambda row: convert_grade(row["Grade"], row["Grade Level"]), axis=1)


<b>Export filtered and converted data:</b><br>
Saves the filtered DataFrame (filtered_data) to a new CSV file (Stored_Grades.csv) using to_csv.
Sets index=False to avoid saving the row index as a separate column.

In [20]:
# Replace "filtered_data.csv" with your desired output filename, set index=False to avoid saving the row index
filtered_data.to_csv(output_path, index=False)

filtered_data.head()

,School Year,Termbinsname,Schoolname,Schoolid,Studentlastname,Studentfirstname,Student Number,State Studentnumber,Grade Level,Course Number,Course Name,Sectionid,Credit Type,Teacher,Teacherlastname,Teacherfirstname,Percent,Grade
0,2023-2024,Y1,Charles Boehm Middle School,255,Pringle,Anthony,1600101,5066239294,8,MATH180,Math 180,29616,MA,3359,Dale,Ashley,90.66,A
1,2023-2024,Y1,Charles Boehm Middle School,255,Roberts,Mikayla,1700142,9613941533,7,MATH180,Math 180,27884,MA,3359,Dale,Ashley,94.59,A
2,2023-2024,Y1,Charles Boehm Middle School,255,Hess,McKenzie,1700217,6513576563,7,MATH180,Math 180,27976,MA,3359,Dale,Ashley,89.53,A
3,2023-2024,Y1,Charles Boehm Middle School,255,Minton,Kaitlyn,1800473,5166477251,6,MATH6,Math 6th,28343,MA,2532,Murphy,David,94.17,A
4,2023-2024,Y1,Charles Boehm Middle School,255,Marota,Mira,1800778,6211658575,8,MATH8,Math 8th,28346,MA,3806,Diamond,Debra,93.89,A
